In [ ]:
source("Main.R")
source("Utilities.R")
library(ArchR)



In [ ]:
install.packages("ggpubr")

In [ ]:
install.packages("BiocManager")
BiocManager::install(version = "3.20")  # matches R 4.4.x
BiocManager::install("rhdf5", ask = FALSE)
remotes::install_github("GreenleafLab/ArchR", upgrade="never", dependencies=TRUE)


In [ ]:
if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager")
if (!requireNamespace("remotes", quietly = TRUE)) install.packages("remotes")

# ArchR needs a bunch of CRAN + Bioconductor deps; these cover most failures
BiocManager::install(c("rhdf5", "SummarizedExperiment", "GenomicRanges", "IRanges",
                       "S4Vectors", "Biostrings", "GenomeInfoDb", "BiocGenerics"),
                     ask = FALSE, update = TRUE)

install.packages(c("data.table", "Matrix", "Rcpp", "ggplot2", "magrittr", "patchwork"),
                 dependencies = TRUE)

# Now install ArchR from GitHub
remotes::install_github("GreenleafLab/ArchR", upgrade = "never", dependencies = TRUE)


In [ ]:
R.version.string


In [ ]:
options(repos = c(CRAN = "https://cloud.r-project.org"))

# Bioconductor packages
install.packages("BiocManager")
BiocManager::install(c(
    "ComplexHeatmap",
    "sparseMatrixStats"
))

# CRAN packages
install.packages(c(
    "ggrepel",
    "gridExtra",
    "Seurat",
    "SeuratObject"
))

# GitHub-only packages
install.packages("remotes")
remotes::install_github("immunogenomics/harmony")
remotes::install_github("immunogenomics/presto")
remotes::install_github("GreenleafLab/chromVARmotifs")

# devtools — try it, but ArchR can work without it
try(install.packages("devtools"))

# Now install ArchR
remotes::install_github("GreenleafLab/ArchR", ref = "master", dependencies = FALSE)

In [ ]:
# This is the most reliable path on problematic systems
conda create -n archr -c conda-forge -c bioconda \
    r-base=4.3 \
    bioconductor-rhdf5 \
    bioconductor-genomicranges \
    bioconductor-summarizedexperiment \
    bioconductor-rsamtools \
    bioconductor-biostrings \
    bioconductor-chromvar \
    bioconductor-motifmatchr \
    r-matrix r-rcpp r-ggplot2 r-data.table r-magrittr r-uwot r-nabor

conda activate archr

# Then install just ArchR (deps already satisfied)
R -e 'install.packages("remotes"); remotes::install_github("GreenleafLab/ArchR", ref="master", dependencies=FALSE)'

In [ ]:
library(ArchR)


In [ ]:
fragments_file <- "../Data/NEPC_sec_screen_ATAC_fragments_for_archR.sorted.tsv.gz"


In [ ]:
# ===========================================================================
# Create an ArchR Project from a fragments file (R.sorted.tsv.gz)
# ===========================================================================
# ArchR expects a fragments file in the format:
#   chr  start  end  barcode  count
# which is the standard 10x fragments.tsv.gz format.
#
# The R.sorted.tsv.gz file should be tabix-indexed (.tbi).
# If not, ArchR will attempt to index it, or you can do it manually.
# ===========================================================================


# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

# Path to your fragments file

# Project name and output directory
project_name <- "MyArchRProject"
output_dir   <- "./ArchROutput"

# Genome: "hg38", "hg19", "mm10", or "mm9"
genome <- "hg38"

# Number of threads
n_threads <- 8

# Seed for reproducibility
set.seed(42)

# Set ArchR threads and genome
addArchRThreads(threads = n_threads)
addArchRGenome(genome)


# ---------------------------------------------------------------------------
# Step 0: Ensure the fragments file is tabix-indexed
# ---------------------------------------------------------------------------
# ArchR requires a .tbi index alongside the .tsv.gz file.
# If R.sorted.tsv.gz.tbi does not exist, create it:

tbi_file <- paste0(fragments_file, ".tbi")
if (!file.exists(tbi_file)) {
    message("Tabix index not found. Indexing fragments file...")
    # Option A: use Rsamtools
    # library(Rsamtools)
    # indexTabix(fragments_file, format = "bed")

    # Option B: use system tabix (requires samtools/htslib installed)
    system(paste("tabix -p bed", fragments_file))
    message("Indexing complete.")
} else {
    message("Tabix index found.")
}


# ---------------------------------------------------------------------------
# Step 1: Create Arrow Files
# ---------------------------------------------------------------------------
# Arrow files are ArchR's on-disk representation of single-cell data.
# Each fragments file becomes one Arrow file.

message("Creating Arrow files...")

ArrowFiles <- createArrowFiles(
    inputFiles = fragments_file,
    sampleNames = project_name,
    minTSS = 4,            # minimum TSS enrichment score to keep a cell
    minFrags = 1000,        # minimum number of fragments to keep a cell
    addTileMat = TRUE,      # add 500bp tile matrix (for LSI/clustering)
    addGeneScoreMat = TRUE, # add gene activity score matrix
    force = TRUE            # overwrite if Arrow file already exists
)

message(paste("Arrow files created:", paste(ArrowFiles, collapse = ", ")))


# ---------------------------------------------------------------------------
# Step 2: Infer doublets (optional but recommended)
# ---------------------------------------------------------------------------

message("Inferring doublets...")

doubScores <- addDoubletScores(
    input = ArrowFiles,
    k = 10,
    knnMethod = "UMAP",
    LSIMethod = 1
)


# ---------------------------------------------------------------------------
# Step 3: Create ArchR Project
# ---------------------------------------------------------------------------

message("Creating ArchR project...")

proj <- ArchRProject(
    ArrowFiles = ArrowFiles,
    outputDirectory = output_dir,
    copyArrows = TRUE       # copy Arrow files into the project directory
)

# Print project summary
print(proj)
message(paste("Cells in project:", nCells(proj)))
message(paste("Samples:", paste(unique(proj$Sample), collapse = ", ")))


# ---------------------------------------------------------------------------
# Step 4: Filter doublets
# ---------------------------------------------------------------------------

message("Filtering doublets...")
proj <- filterDoublets(proj)
message(paste("Cells after doublet filtering:", nCells(proj)))


# ---------------------------------------------------------------------------
# Step 5: Basic QC plots
# ---------------------------------------------------------------------------

# TSS enrichment vs log10(unique fragments) — Ridge plot
p1 <- plotGroups(
    ArchRProj = proj,
    groupBy = "Sample",
    colorBy = "cellColData",
    name = "TSSEnrichment",
    plotAs = "ridges"
)

# Fragment size distribution
p2 <- plotFragmentSizes(ArchRProj = proj)

# TSS enrichment profile
p3 <- plotTSSEnrichment(ArchRProj = proj)

# Save QC plots
plotPDF(p1, p2, p3,
        name = "QC_Plots",
        ArchRProj = proj,
        addDOC = FALSE,
        width = 5, height = 5)

message("QC plots saved to project Plots directory.")


# ---------------------------------------------------------------------------
# Step 6: Dimensionality reduction and clustering
# ---------------------------------------------------------------------------

# LSI (Latent Semantic Indexing) on tile matrix
pathToMacs2 <- findMacs2()

# ---------------------------------------------------------------------------
# Step 7: Save project
# ---------------------------------------------------------------------------

message("Saving ArchR project...")
saveArchRProject(
    ArchRProj = proj,
    outputDirectory = output_dir,
    load = FALSE
)

message(paste("ArchR project saved to:", output_dir))
message("Done!")

In [ ]:
if (!requireNamespace("devtools", quietly = TRUE)) install.packages("devtools")


In [ ]:
devtools::install_github("GreenleafLab/ArchR", ref="master", repos = BiocManager::repositories())
